In [8]:
# Import Libraries

import os
import re
import pandas as pd
from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [9]:
#Load API keys

load_dotenv()

# Connect to the database

DB_USER = os.getenv("pg_user")
DB_HOST = os.getenv("pg_host")
DB_PORT = os.getenv("pg_port")
DB_PASSWORD = os.getenv("pg_password")

DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres"

try:
    db = SQLDatabase.from_uri(DB_URL, schema="public" , include_tables=['customer_segments' , 'transactions'])
    print("Connected to the database successfully.")

except Exception as e:
    print(f"Failed to connect to the database: {e}")



Connected to the database successfully.


In [10]:
schema_info = db.get_table_info()
print("Connected Views:", db.get_usable_table_names())
print("\n--- Detected Schema ---\n", schema_info)

Connected Views: ['customer_segments', 'transactions']

--- Detected Schema ---
 
CREATE TABLE public.customer_segments (
	customer_id BIGINT, 
	country TEXT, 
	recency DOUBLE PRECISION, 
	frequency BIGINT, 
	monetary DOUBLE PRECISION, 
	cluster INTEGER, 
	segment_name TEXT
)

/*
3 rows from customer_segments table:
customer_id	country	recency	frequency	monetary	cluster	segment_name
12346	United Kingdom	326.0	1	77183.6	0	Loyal Buyers
12347	Iceland	2.0	7	4309.999999999998	1	Recent/Promising Buyers
12348	Finland	75.0	4	1797.24	1	Recent/Promising Buyers
*/


CREATE TABLE public.transactions (
	"Description" TEXT, 
	"Quantity" BIGINT, 
	"InvoiceDate" TIMESTAMP WITHOUT TIME ZONE, 
	"UnitPrice" DOUBLE PRECISION, 
	"CustomerID" BIGINT, 
	"Country" TEXT
)

/*
3 rows from transactions table:
Description	Quantity	InvoiceDate	UnitPrice	CustomerID	Country
WHITE HANGING HEART T-LIGHT HOLDER	6	2010-12-01 08:26:00	2.55	17850	United Kingdom
WHITE METAL LANTERN	6	2010-12-01 08:26:00	3.39	17850	United K

In [11]:
# Define LLM 

llm = ChatGroq(model_name= 'llama-3.3-70b-versatile', temperature= 0, groq_api_key= os.getenv("GROQ_API_KEY"))

In [12]:
# Set up Query Tool and gaurdrails 

def run_query_tool(query: str) -> str:
    """
    Function to run a SQL query using the database connection.
    """
    cleaned_query = query.strip().replace("```sql", "").replace("```", "")

    if not cleaned_query.upper().startswith("SELECT") and not cleaned_query.upper().startswith("WITH"):
        return "ERROR: Only read-only SELECT queries are allowed."

    forbidden_keywords = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE"]
    if any(keyword in cleaned_query.upper() for keyword in forbidden_keywords):
        return 'ERROR: FORBIDDEN KEYWORD DETECTED. Only read-only SELECT queries are allowed.'

    try: 
        df = pd.read_sql(cleaned_query, db._engine)
        return True, df
    except Exception as e:
        return False, f'ERROR: An error occurred while executing the query: {str(e)}'

In [ ]:
# Set up the system prompt and loop chain for sql query generation and execution

system_prompt = f"""
You are an expert PostgreSQL Data Analyst. 
Given a user question, write a syntactically correct PostgreSQL query using ONLY the views available below.

### DATABASE SCHEMA:
{schema_info}

### INSTRUCTIONS:
1. Return ONLY the raw SQL query. Do not wrap it in markdown block quotes or extra conversational text.
2. Use standard PostgreSQL syntax.
3. For text filters, use ILIKE for case-insensitive matching.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}\n\nPrevious error (if any): {error}")
])

chain = prompt_template | llm | StrOutputParser()

def ask_agent(question: str, max_retries: int = 3):
    print(f" Question: {question}\n")
    error_context = "None"
    
    for attempt in range(1, max_retries + 1):
        generated_sql = chain.invoke({"question": question, "error": error_context})
        print(f" [Attempt {attempt}] Generated SQL:\n{generated_sql}\n")
        
        success, result = run_query_tool(generated_sql)
        
        if success:
            print(" Execution Successful!\n")
            return result  # Returns Pandas DataFrame
        else:
            print(f" {result}. Retrying...\n")
            error_context = result

    print(" Reached max retries.")
    return None

In [19]:
# Run test and display interactive pandas table directly in Jupyter
df_result = ask_agent("What is the average recency and total monetary value across customer_rfm?")
display(df_result)

 Question: What is the average recency and total monetary value across customer_rfm?

 [Attempt 1] Generated SQL:
SELECT AVG(recency) AS average_recency, SUM(monetary) AS total_monetary FROM customer_segments

 Execution Successful!



,average_recency,total_monetary
0,92.536422,8911407.904
